# 13 · Selección de variables, regularización y reducción de ruido

Más features no siempre significa mejor modelo. Variables irrelevantes, redundantes o filtradas desde el futuro aumentan variance, costo y fragilidad.

## Objetivos
- Diferenciar filter, wrapper y embedded methods.
- Usar correlación, mutual information, chi² y ANOVA con criterio.
- Aplicar RFE/RFECV.
- Usar L1 y árboles como métodos embedded.
- Comparar selección supervisada vs PCA.
- Entender estabilidad de features y leakage.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif, RFE, RFECV, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
SEED=42
X,y=make_classification(n_samples=2500,n_features=100,n_informative=10,n_redundant=20,n_repeated=5,random_state=SEED,shuffle=False)
cols=[f'x{i:03}' for i in range(X.shape[1])]; X=pd.DataFrame(X,columns=cols)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)

## 1. Filter methods
Evalúan cada variable antes del modelo. Son rápidos y útiles en dimensionalidad alta.
- correlación: relación lineal numérica;
- ANOVA F: separación lineal entre clases;
- chi²: asociación para features no negativas/categóricas codificadas;
- mutual information: dependencia no lineal más general.

Limitación: un filtro univariado puede perder variables que solo son útiles mediante interacciones.


In [ ]:
mi=mutual_info_classif(Xtr,ytr,random_state=SEED); f,_=f_classif(Xtr,ytr)
rank=pd.DataFrame({'feature':cols,'mutual_info':mi,'F':f}).sort_values('mutual_info',ascending=False); display(rank.head(20))

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=SEED); rows=[]
for k in [5,10,20,40,70,100]:
 pipe=Pipeline([('select',SelectKBest(mutual_info_classif,k=k)),('scale',StandardScaler()),('model',LogisticRegression(max_iter=3000))])
 rows.append([k,cross_val_score(pipe,Xtr,ytr,cv=cv,scoring='roc_auc',n_jobs=-1).mean()])
pd.DataFrame(rows,columns=['k','ROC_AUC']).round(4)

## 2. Wrapper methods: RFE
RFE entrena repetidamente un estimador y elimina variables menos importantes. Puede capturar información condicionada al modelo, pero es mucho más costoso. RFECV incorpora CV para elegir el número de variables.


In [ ]:
# ejemplo reducido para que Colab no tarde demasiado
base=LogisticRegression(max_iter=3000)
rfe=RFECV(base,step=10,cv=4,scoring='roc_auc',min_features_to_select=5,n_jobs=-1).fit(StandardScaler().fit_transform(Xtr),ytr)
print('n seleccionadas',rfe.n_features_)

## 3. Embedded: Lasso/L1
La penalización L1 puede llevar coeficientes a cero durante el entrenamiento. Es eficiente y naturalmente acoplada al modelo lineal. Con features altamente correlacionadas puede elegir una de forma inestable.


In [ ]:
for C in [.01,.05,.1,.5,1]:
 pipe=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(penalty='l1',solver='liblinear',C=C,max_iter=3000))]).fit(Xtr,ytr)
 coef=pipe[-1].coef_.ravel(); print('C',C,'no-cero',np.sum(coef!=0),'score',pipe.score(Xte,yte))

## 4. Embedded con árboles
`SelectFromModel(RandomForest)` selecciona según importancias, pero recuerda el sesgo de impurity importance. Para selección más confiable, permutation importance dentro de CV es mejor aunque más costosa.


In [ ]:
rf=RandomForestClassifier(n_estimators=400,random_state=SEED,n_jobs=-1).fit(Xtr,ytr)
sel=SelectFromModel(rf,prefit=True,threshold='median'); print('features seleccionadas',sel.get_support().sum())

## 5. Selección vs PCA
Feature selection conserva variables originales, importante para interpretabilidad y serving. PCA construye combinaciones lineales y puede comprimir redundancia incluso si ninguna feature es claramente descartable. PCA no mira el target; selección supervisada sí.

## 6. Estabilidad
Si las features elegidas cambian drásticamente con cada seed/fold, la explicación es frágil. En problemas de política, riesgo o ciencia, mide estabilidad de selección con bootstrap.

## Errores comunes
- selección usando todo el dataset antes de CV;
- eliminar features correlacionadas solo por un umbral arbitrario;
- creer que feature importance = causalidad;
- seleccionar miles de features sin controlar multiple testing;
- descartar una feature sensible sin analizar proxies.

## Ejercicios
1. Identifica las 10 informativas originales del dataset sintético.
2. Compara MI, ANOVA, L1 y RF.
3. Mide estabilidad por 30 bootstraps.
4. Crea dos features altamente correlacionadas y observa L1.
5. Compara top-20 selección vs PCA 20 componentes.
6. Diseña selección temporal evitando usar estadísticas futuras.
